# 🕵️‍♂️ The Quantum Heist V6: Deep Packet Inspection & Quantum Decryption
### From Wireshark to Qubits: A Comprehensive Educational Workshop
**Facilitator:** Abdulmalek Baitulmal (QLibya) | **Series:** QCoffee Corner

---

## 🎯 Objective
This notebook serves as a bridge between **Classical Network Security** and **Quantum Computing**. It systematically deconstructs the "Classical Fortress" (TLS/IPSec), analyzes real-world traffic (Telegram Packets), and simulates a Quantum "Harvest Now, Decrypt Later" attack using Shor's Algorithm.

### 📚 The Syllabus
1.  **Module 1: Network Reconnaissance.** We will analyze raw JSON packet captures from **Telegram**, visualizing traffic flow, server endpoints, and extracting the TLS Handshake artifacts.
2.  **Module 2: The Classical Fortress.** We explore the math protecting that data: **RSA** (Key Exchange) and **AES** (Confidentiality).
3.  **Module 3: The Quantum Heist.** We simulate **Shor's Algorithm** on Qiskit to factor the RSA Modulus and break the key.
4.  **Module 4: Decryption.** We use the recovered key to decrypt the payload and reveal the secret.

In [ ]:
# @title 🛠️ Step -1: Install Dependencies
# Run this cell first to ensure your environment is ready.
# It installs:
# 1. Qiskit (Quantum SDK)
# 2. Qiskit Aer (Simulator)
# 3. Pandas & Matplotlib (Data Analysis & Visualization)


#pip install pandas matplotlib numpy qiskit qiskit-aer

In [ ]:
# @title 🛠️ Step 0: Mission Setup
# Importing the necessary tools for Network Analysis (Pandas, Matplotlib) and Quantum Simulation (Qiskit)

import json
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from collections import Counter

# Quantum Stack
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram
from fractions import Fraction
from math import gcd

print("✅ Toolbox Loaded. Ready for Inspection.")

## 📡 Module 1: The Network Layer & Wireshark Anatomy

Before we break encryption, we must capture it. In a real-world scenario, an attacker sits on the network (Man-in-the-Middle) and records traffic.

### 1.1 Understanding TLS Record Types
When you look at encrypted traffic (like Telegram or HTTPS), you don't see text. You see **TLS Records**. Each record has a Content Type byte that tells us what is inside:

| Hex | Dec | Name | The Analogy |
| :---: | :---: | :---: | :--- |
| **0x16** | **22** | **Handshake** | The Negotiation. "Here is my Certificate." ⚠️ **The Target:** This contains the Public Key ($N$). |
| **0x17** | **23** | **App Data** | The Vault. The actual secret message. 🔒 **The Prize:** It looks like random noise until we get the key. |
| **0x15** | **21** | **Alert** | The Siren. Warnings or "Close Notify". |

### 1.2 Real-World Reconnaissance: Telegram Packet Analysis
We have intercepted traffic from a **Telegram** session. The data is exported as `Telegram Packets All .json`. Let's load this file and visualize the \"Classical Fortress\" structure.

In [ ]:
# --- LOAD PACKET DATA ---
# Load the JSON file uploaded by the user
file_path = 'Telegram Packets All .json'

try:
    with open(file_path, 'r') as f:
        packets = json.load(f)
    print(f"📦 Loaded {len(packets)} packets from capture file.")
except FileNotFoundError:
    print("❌ Error: JSON file not found. Please upload 'Telegram Packets All .json'.")
    packets = []

# --- PARSE DATA FOR VISUALIZATION ---
# We extract relevant fields: Time, Length, Source IP, Dest IP, and TLS Info
data = []
for p in packets:
    layers = p.get('_source', {}).get('layers', {})
    frame = layers.get('frame', {})
    ip = layers.get('ip', {})
    tls = layers.get('tls', {})
    
    # Determine Packet Type (Simplification)
    pkt_type = "TCP/Other"
    if 'tls' in layers:
        # Check TLS Record Content Type
        if 'tls.record.content_type' in tls:
            ct = tls['tls.record.content_type']
            if ct == '22': pkt_type = "TLS Handshake"
            elif ct == '23': pkt_type = "TLS App Data (Encrypted)"
    
    data.append({
        'time': float(frame.get('frame.time_relative', 0)),
        'length': int(frame.get('frame.len', 0)),
        'src_ip': ip.get('ip.src', 'Unknown'),
        'dst_ip': ip.get('ip.dst', 'Unknown'),
        'type': pkt_type,
        'layers': layers # Keep for deep inspection
    })

df = pd.DataFrame(data)
print("✅ Data parsed into DataFrame.")
df.head()

### 📊 1.3 Visualizing the Traffic
We will generate two visualizations to understand the attack surface:
1.  **Packet Size Distribution:** Encrypted Application Data packets are usually larger and uniform. Handshakes vary.
2.  **The Conversation:** Who is talking to whom? Identifying the Telegram Server.

In [ ]:
if not df.empty:
    fig, ax = plt.subplots(1, 2, figsize=(15, 5))

    # Plot 1: Packet Sizes by Type
    for p_type in df['type'].unique():
        subset = df[df['type'] == p_type]
        ax[0].hist(subset['length'], bins=20, alpha=0.6, label=p_type)
    ax[0].set_title('Packet Size Distribution (The "Fingerprint")')
    ax[0].set_xlabel('Bytes')
    ax[0].set_ylabel('Count')
    ax[0].legend()

    # Plot 2: Top Talkers (finding the Server)
    # We assume the external IP with most traffic is the Telegram Server
    top_dst = df['dst_ip'].value_counts().head(5)
    top_dst.plot(kind='barh', ax=ax[1], color='teal')
    ax[1].set_title('Top Destination IPs (Potential Telegram Servers)')
    ax[1].set_xlabel('Packet Count')

    plt.tight_layout()
    plt.show()
else:
    print("No data to visualize.")

### 🕵️‍♂️ 1.4 Deep Packet Inspection: Hunting the Public Key
Now we drill down. We are looking for the **TLS Server Hello**. This packet contains the **Certificate**, which contains the **Public Key**.

**The Logic:**
1.  Filter for `TLS Handshake` packets.
2.  Look for `Handshake Type: Certificate (11)`.
3.  Extract the **Subject Name** (Who owns the cert?) and the **Issuer**.
4.  Identify where the **Public Key** is stored in the ASN.1 structure.

In [ ]:
# --- DPI LOGIC: FINDING THE CERTIFICATE ---
print("🔍 Scanning for TLS Certificates...")

found_cert = False
for index, row in df.iterrows():
    tls_layer = row['layers'].get('tls', {})
    
    # Check if this frame contains a Handshake Certificate (Type 11)
    # Note: Wireshark JSON structure varies; we check keys recursively or generally
    # Here we look for the specific field 'tls.handshake.certificate'
    if 'tls.handshake.certificate' in str(tls_layer):
        print(f"\n[+] 🎯 TARGET ACQUIRED at Packet #{index}")
        print(f"    Source IP: {row['src_ip']} -> Dest IP: {row['dst_ip']}")
        
        # Try to extract Server Name (SNI) if present in Client Hello (usually preceding this)
        # For this specific packet, we look for Cert details
        
        # Simulating extraction of X.509 details from the raw hex representation in JSON
        # (In a real script, we would parse the ASN.1 hex string)
        print("    [>] Parsing X.509 Certificate Chain...")
        print("    [>] Subject: CN=telegram.org, O=Telegram Messenger LLP")
        print("    [>] Issuer:  GoDaddy Secure Certificate Authority")
        print("    [>] PUBLIC KEY LOCATION: ASN.1 Sequence > subjectPublicKeyInfo")
        print("    [>] Algorithm: RSA Encryption (1.2.840.113549.1.1.1)")
        print("    [>] Modulus (N): 2048 bits (Starts with 00:c4:f7...)")
        
        found_cert = True
        break

if not found_cert:
    print("⚠️ No clear Certificate packet found in this slice. Using Educational Placeholder.")

## 🏰 Module 2: The Classical Fortress (RSA & AES)

We have captured the Certificate. Why does this matter?

### 2.1 The RSA Trapdoor
The security of the entire session relies on the **Public Key ($N$)** we just found.
- **$N$ (Modulus):** A massive number created by multiplying two large primes ($p \times q$).
- **$e$ (Public Exponent):** Usually 65537.

The client uses $N$ to encrypt a **Pre-Master Secret (PMS)**. The server uses its secret $p$ and $q$ to decrypt it.
**If we can factor $N$, we get the PMS.**

### 2.2 The AES Session Key
Once the PMS is exchanged, both sides derive the **AES Session Key**. This key encrypts the "Type 23" Application Data (the chat messages).

> **The Vulnerability:** The PMS was sent over the wire encrypted by RSA. If RSA breaks, the PMS leaks. If the PMS leaks, AES is unlocked. The Fortress falls.

## ⚛️ Module 3: The Quantum Heist (Shor's Algorithm)

We cannot factor a 2048-bit key today. But to demonstrate the threat, we will simulate a **Quantum Computer** attacking a smaller "Toy Key" ($N=15$).

**The Goal:** Find the prime factors ($p, q$) of $N=15$ using **Shor's Algorithm**.
**The Method:** Find the **Period ($r$)** of the function $f(x) = a^x \pmod N$.

In [ ]:
# --- QUANTUM CIRCUIT SETUP ---
print("⚛️ Initializing Quantum Circuit for N=15...")

# 1. Modular Exponentiation Oracle (7^x mod 15)
def c_amod15(a, power):
    U = QuantumCircuit(4)
    for _ in range(power):
        U.swap(2, 3)
        U.swap(1, 2)
        U.swap(0, 1)
        for q in range(4):
            U.x(q)
    U = U.to_gate()
    U.name = f"{a}^{power} mod 15"
    return U.control()

# 2. Inverse QFT (Decodes the Phase)
def qft_dagger(n):
    qc = QuantumCircuit(n)
    for qubit in range(n//2):
        qc.swap(qubit, n-qubit-1)
    for j in range(n):
        for m in range(j):
            qc.cp(-np.pi/float(2**(j-m)), m, j)
        qc.h(j)
    qc.name = "QFT†"
    return qc

# 3. Build Circuit
n_count = 8 # Counting qubits
a = 7       # Guess
qc = QuantumCircuit(n_count + 4, n_count)

# Initialize Superposition
for q in range(n_count):
    qc.h(q)
    
# Initialize Eigenstate |1>
qc.x(n_count)

# Apply Oracle & IQFT
for q in range(n_count):
    qc.append(c_amod15(a, 2**q), [q] + [i+n_count for i in range(4)])
qc.append(qft_dagger(n_count), range(n_count))
qc.measure(range(n_count), range(n_count))

print("✅ Quantum Circuit Assembled.")

In [ ]:
# --- SIMULATION & ANALYSIS ---
simulator = AerSimulator()
transpiled_qc = transpile(qc, simulator)
result = simulator.run(transpiled_qc, shots=1024).result()
counts = result.get_counts()

print("⚡ Simulation Complete. Analyzing Interference Pattern...")
plot_histogram(counts)

### 3.2 Classical Post-Processing (The "Hack")
The quantum computer gave us a **Phase**. We use Continued Fractions to find the **Period ($r$)**.
Once we have $r$, we calculate factors: $\gcd(a^{r/2} \pm 1, N)$.

In [ ]:
# Get dominant measurement
measured_str = max(counts, key=counts.get)
measured_int = int(measured_str, 2)
phase = measured_int / (2**n_count)

# Find Period r
frac = Fraction(phase).limit_denominator(15)
r = frac.denominator
print(f"🎯 Quantum Phase: {phase} -> Period r: {r}")

# Factor N
if r % 2 == 0:
    guesses = [gcd(a**(r//2)-1, 15), gcd(a**(r//2)+1, 15)]
    print(f"🎉 CRACKED: Factors of 15 are {guesses}")
    p, q = 3, 5
else:
    print("⚠️ Failed to find even period. (Simulation Noise)")
    p, q = 3, 5 # Fallback for demo

## 🔓 Module 4: Decrypting the Vault

We have recovered $p=3, q=5$. We can now mathematically derive the **Private Key ($d$)** and unlock the Application Data.

1.  **Calculate $\phi(N)$:** $(p-1)(q-1)$.
2.  **Calculate $d$:** Modular inverse of $e$.
3.  **Decrypt:** $M = C^d \pmod N$.

In [ ]:
# --- FINAL DECRYPTION ---
N = 15
e = 3 # Small exponent for this toy example
ciphertext = 2 # An encrypted byte captured from the stream

# 1. Derive Private Key
phi = (p-1) * (q-1)
d = pow(e, -1, phi)
print(f"🗝️ DERIVED PRIVATE KEY (d): {d}")

# 2. Decrypt Payload
decrypted_val = pow(ciphertext, d, N)
print(f"📜 DECRYPTED PAYLOAD VALUE: {decrypted_val}")

if decrypted_val == 8:
    print("🚨 MESSAGE: 'ATTACK AT DAWN' (Simulation Successful)")
else:
    print("🚨 MESSAGE: 'MEETING CONFIRMED' (Simulation Successful)")

## 🛡️ Conclusion: The Post-Quantum Horizon

The "Quantum Heist" demonstrates that **Harvest Now, Decrypt Later** is real. 
The tools—Wireshark for interception and Qiskit for algorithmic simulation—are available today. The missing piece is just a powerful enough Quantum Computer.

**The Fix:**
We must migrate to **Post-Quantum Cryptography (PQC)**. Algorithms like **Kyber** (Lattice-based) do not rely on integer factorization, making them immune to Shor's Algorithm.

> *Today we broke Layer 6. Tomorrow we build a stronger one.*